# FTWP Chapter 1 — Review and Exercises

This notebook consolidates the **six non-capstone tasks** for FTWP Chapter 1:
three review exercises and three regular exercises. The problem statements are
paraphrased from the CPF exercise sheet. Numerical parameters are stated where
the source task leaves them open.

The notebook is designed to run from top to bottom in Google Colab or a standard
Python/Jupyter environment.

In [1]:
%matplotlib inline

## Review exercises

### 1. Expected payoff in a two-state model

For a payoff vector $X=(X_u,X_d)$ and Up-state probability $p$,

$$E[X]=pX_u+(1-p)X_d.$$

I use $X=(12,4)$ and $p=0.60$.

### 2. European call payoff

For stock payoff $S_1$ and strike $K$, compute

$$C_1=\max(S_1-K,0)$$

componentwise.

### 3. Portfolio payoff

For holdings $(\phi_B,\phi_S)$, the state-contingent portfolio payoff is

$$V_1=\phi_BB_1+\phi_SS_1.$$

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

states = ["Up", "Down"]

# Review 1
X = np.array([12.0, 4.0])
p = 0.60
EX = p * X[0] + (1.0 - p) * X[1]

# Review 2
S1 = np.array([20.0, 5.0])
K = 15.0
C1 = np.maximum(S1 - K, 0.0)

# Review 3: use the Chapter 2 replicating holdings for this call.
B1 = np.array([11.0, 11.0])
phi = np.array([-5.0 / 33.0, 1.0 / 3.0])
V1 = phi[0] * B1 + phi[1] * S1

review = pd.DataFrame(
    {"X": X, "Stock": S1, "Call": C1, "Portfolio": V1},
    index=states,
)

print(f"E[X] = {EX:.2f}")
display(review)
print("Portfolio cashflows equal the call payoff:", np.allclose(V1, C1))

E[X] = 8.80


,X,Stock,Call,Portfolio
Up,12.0,20.0,5.0,5.000000e+00
Down,4.0,5.0,0.0,-2.220446e-16


Portfolio cashflows equal the call payoff: True


The expectation is $8.80$. The portfolio pays $5$ in the Up state and $0$ in
the Down state, so its two components are cashflows contingent on which state
occurs.

## Regular exercises

### 4. A portfolio that costs exactly one today

With current prices $B_0=S_0=10$, the budget equation is

$$10\phi_B+10\phi_S=1.$$

There are infinitely many solutions. I choose equal current allocations,
$\phi_B=\phi_S=0.05$.

### 5. Exact replication

Let $M=[B_1\;S_1]$. Solve $M\phi=C_1$ and verify the residual.

### 6. Sensitivity to a payoff perturbation

Increase the stock payoff in the Up state from $20$ to $21$, hold current
prices fixed, and recompute the hedge for the same target claim.

In [3]:
B0 = 10.0
S0 = 10.0
M0 = np.array([B0, S0])
M = np.column_stack((B1, S1))

# Exercise 4
phi_unit_cost = np.array([0.05, 0.05])
unit_cost = M0 @ phi_unit_cost
unit_payoff = M @ phi_unit_cost

# Exercise 5
phi_exact = np.linalg.solve(M, C1)
replication_residual = M @ phi_exact - C1
replication_price = M0 @ phi_exact

# Exercise 6
S1_perturbed = np.array([21.0, 5.0])
M_perturbed = np.column_stack((B1, S1_perturbed))
phi_perturbed = np.linalg.solve(M_perturbed, C1)
price_perturbed = M0 @ phi_perturbed

sensitivity = pd.DataFrame(
    {
        "Original": phi_exact,
        "Perturbed Up payoff": phi_perturbed,
        "Change": phi_perturbed - phi_exact,
    },
    index=["Bond holding", "Stock holding"],
)

print("Unit-cost portfolio cost:", unit_cost)
print("Unit-cost portfolio payoff:", unit_payoff)
print()
print("Exact replication holdings:", phi_exact)
print("Replication price:", replication_price)
print("Residual:", replication_residual)
display(sensitivity)
print("Perturbed replication price:", price_perturbed)

Unit-cost portfolio cost: 1.0
Unit-cost portfolio payoff: [1.55 0.8 ]

Exact replication holdings: [-0.15151515  0.33333333]
Replication price: 1.8181818181818183
Residual: [0.0000000e+00 4.4408921e-16]


,Original,Perturbed Up payoff,Change
Bond holding,-0.151515,-0.142045,0.009470
Stock holding,0.333333,0.312500,-0.020833


Perturbed replication price: 1.7045454545454544


The perturbation changes both hedge weights because the columns of the payoff
matrix define the available state-by-state cashflow directions. With a larger
Up-state stock payoff, fewer stock units are needed to create the call's Up-state
payment, and the bond position adjusts to keep the Down-state payoff at zero.

In [4]:
checks = {
    "expectation": np.isclose(EX, 8.8),
    "call payoff": np.allclose(C1, [5.0, 0.0]),
    "portfolio cashflows": np.allclose(V1, C1),
    "unit current cost": np.isclose(unit_cost, 1.0),
    "exact replication": np.allclose(M @ phi_exact, C1),
    "zero residual": np.allclose(replication_residual, 0.0),
    "perturbed replication": np.allclose(M_perturbed @ phi_perturbed, C1),
}

for name, passed in checks.items():
    print(f"{name}: {passed}")
assert all(checks.values())

expectation: True
call payoff: True
portfolio cashflows: True
unit current cost: True
exact replication: True
zero residual: True
perturbed replication: True
